# Главный результат за один запуск

Двадцать задач, которых ученик при разработке не видел (наборы S и Z: записи из кодов и чисел, написаны двадцатью независимыми авторами после заморозки кода). Два участника на одних данных:

- **ученик** (`records.py`): точный грунт из простых ходов, сеть-подсказчик в 12 тысяч весов (`prior.json`) и спрашивание — учится на 20 эпизодах и ещё до 20 запрашивает сам. Считает на процессоре;
- **трансформер** (`colab/baseline.py`, 2,1 млн весов) с нуля на 20 и на 200 эпизодах.

Проверка — на эпизодах обучающего размера и в 3 и 10 раз длиннее. На процессоре всё занимает около полутора часов, с видеокартой — около получаса (её использует только трансформер). Короткая проверка, что всё запускается: `%env MX_QUICK=1` до первой ячейки.


In [ ]:
import os, subprocess, sys, json, re, time
ROOT = os.environ.get('MX_ROOT', '/content')
QUICK = bool(os.environ.get('MX_QUICK'))
WORK = ROOT + '/mx-demo'
os.makedirs(WORK, exist_ok=True)
if os.path.exists(ROOT + '/MX-Project'):
    subprocess.run(['git', '-C', ROOT + '/MX-Project', 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/kireikosasha/MX-Project', ROOT + '/MX-Project'], check=True)
M5 = ROOT + '/MX-Project/research_m5'
os.environ['PYTHONHASHSEED'] = '0'
Q = ' --train 60 --test 20' if QUICK else ''
for name in 'SZ':
    !python {M5}/newtasks.py --set {name} --out {WORK}/new/{name}{Q} | tail -1
    for size in (3, 10):
        !python {M5}/newtasks.py --set {name} --size {size} --train 20 --out {WORK}/new/{name}{size}{' --test 20' if QUICK else ''} | tail -1


In [ ]:
# трансформер — в фоне, оба набора подряд
B = sys.executable + ' -u ' + M5 + '/colab/baseline.py' + (' --minutes 0.2' if QUICK else '')
cmd = '; '.join(f"{B} --data {WORK}/new/{n} --lang '' --sizes {'20' if QUICK else '20,200'} --sents 150 --also {WORK}/new/{n}3,{WORK}/new/{n}10 --out {WORK}/transformer_{n}.json" for n in 'SZ')
base = subprocess.Popen(cmd, shell=True, stdout=open(WORK + '/transformer.txt', 'w'), stderr=subprocess.STDOUT)
print('трансформер запущен в фоне')


In [ ]:
# ученик: 20 эпизодов, спрашивание, подсказка сети; два набора параллельно
t0 = time.time()
T = ' --tasks 1,2 --limit 40000' if QUICK else ''
jobs = [subprocess.Popen(f"{sys.executable} -u {M5}/records.py --data {WORK}/new/{n} --ask 20 --hint {M5}/prior.json --show --also {WORK}/new/{n}3,{WORK}/new/{n}10{T} > {WORK}/learner_{n}.txt 2>&1", shell=True) for n in 'SZ']
for j in jobs:
    j.wait()
print(f'ученик: {time.time() - t0:.0f} с')
for n in 'SZ':
    print('набор', n)
    for l in open(f'{WORK}/learner_{n}.txt'):
        if re.match(r'(task|test +\d+:)', l):
            print('  ', l.strip()[:150])


In [ ]:
base.wait()
def learner(path):
    rows, asked, work = {1: [], 3: [], 10: []}, [], 0
    for l in open(path):
        m = re.match(r'test +\d+(?: in \w*?(\d+))?: (\d+)/(\d+)', l)
        if m:
            rows[int(m.group(1) or 1)].append(100 * int(m.group(2)) / int(m.group(3)))
        m = re.search(r'asked for (\d+) more', l)
        if m:
            asked.append(int(m.group(1)))
        m = re.search(r'(\d+) episodes played', l)
        if m:
            work += int(m.group(1))
    return rows, asked, work
def mean(v):
    return sum(v) / len(v) if v else float('nan')
print('доля верных ответов, %, среднее по задачам; размер тестовых эпизодов 1 / 3 / 10')
for n in 'SZ':
    rows, asked, work = learner(f'{WORK}/learner_{n}.txt')
    print(f"{n}: ученик, 20 эпизодов и в среднем {mean(asked):.0f} запрошенных:  " + ' / '.join(f'{mean(rows[s]):.1f}' for s in (1, 3, 10)) +
          f"   задач до 95 % и выше: {sum(x >= 95 for x in rows[1])} из {len(rows[1])}   сыграно эпизодов: {work}")
    if os.path.exists(f'{WORK}/transformer_{n}.json'):
        for e in json.load(open(f'{WORK}/transformer_{n}.json')):
            f = lambda t: mean([100 * x[1] / x[2] for x in t])
            print(f"{n}: трансформер, {e['size']} эпизодов:  " + ' / '.join(f'{f(t):.1f}' for t in [e['test']] + list(e['also'].values())) +
                  f"   задач до 95 % и выше: {sum(100 * x[1] / x[2] >= 95 for x in e['test'])} из {len(e['test'])}")


## Необязательно: тот же трансформер на 0,2 млрд весов

Исходная идея называет размер — порядка 0,2 млрд параметров. У ученика выше обучаемых весов двенадцать тысяч (сеть-подсказчик), остальное — грунт. Эта ячейка проверяет обратную сторону: закрывает ли один только размер разрыв без грунта. Тот же трансформер, увеличенный до 203 млн весов (ширина 1024, 16 слоёв, 16 голов), учится с нуля на тех же 20 и 200 эпизодах по тому же рецепту, что и маленький, и проверяется на тех же тестах. Нужна видеокарта: на T4 это два-три часа (до 45 минут на одно обучение). Без видеокарты ячейка себя пропускает; в короткой проверке (`MX_QUICK=1`) она идёт с крошечной сетью, только чтобы убедиться, что всё запускается.


In [ ]:
import torch
if torch.cuda.is_available() or QUICK:
    BIG = ' --width 64 --layers 2 --heads 4 --batch 16 --minutes 0.2' if QUICK else ' --width 1024 --layers 16 --heads 16 --batch 16 --minutes 45'
    for n in 'SZ':
        !{sys.executable} -u {M5}/colab/baseline.py --data {WORK}/new/{n} --lang '' --sizes {'20' if QUICK else '20,200'} --sents 150 --also {WORK}/new/{n}3,{WORK}/new/{n}10{BIG} --out {WORK}/transformer_big_{n}.json | grep -E "weights|test"
    print('доля верных ответов, %, среднее по задачам; размер тестовых эпизодов 1 / 3 / 10')
    for n in 'SZ':
        for e in json.load(open(f'{WORK}/transformer_big_{n}.json')):
            f = lambda t: mean([100 * x[1] / x[2] for x in t])
            print(f"{n}: трансформер, {e['weights'] / 1e6:.0f} млн весов, {e['size']} эпизодов:  " + ' / '.join(f'{f(t):.1f}' for t in [e['test']] + list(e['also'].values())) +
                  f"   задач до 95 % и выше: {sum(100 * x[1] / x[2] >= 95 for x in e['test'])} из {len(e['test'])}")
else:
    print('видеокарты нет: сравнение с трансформером на 0,2 млрд весов пропущено')


## Необязательно: язык тем же учеником

Тот же ученик без единой части про слова читает предложение как строку журнала: слова, которые чередуются на одном месте, — вещи, остальное — код операции (`rows.py`). Здесь он учит двенадцать задач шаблонным языком из набора D — их написали двенадцать независимых авторов уже после того, как код был заморожен, — с 20 рассказов и до 20 запрошенных, а трансформер учится на тех же 20 и на 200 рассказах. В слепом прогоне было 88,8 % у ученика при 39,9 % и 48,3 % у трансформера. На двух ядрах Colab ученик считает около двух часов, поэтому часть включается отдельно: `%env MX_LANG=1` до первой ячейки.


In [ ]:
if os.environ.get('MX_LANG') or QUICK:
    !python {M5}/newtasks.py --set D --out {WORK}/new/D{Q} | tail -1
    halves = ('2',) if QUICK else ('1,3,5,7,9,11', '2,4,6,8,10,12')     # две очереди: на Colab два ядра
    t0 = time.time()
    jobs = [subprocess.Popen(f"{sys.executable} -u {M5}/rows.py --data {WORK}/new/D --ask 20 --limit {40000 if QUICK else 300000} --tasks {h} --show > {WORK}/bridge_D_{i}.txt 2>&1", shell=True) for i, h in enumerate(halves)]
    !{sys.executable} -u {M5}/colab/baseline.py --data {WORK}/new/D --lang '' --sizes {'20' if QUICK else '20,200'}{' --minutes 0.2' if QUICK else ''} --out {WORK}/transformer_D.json | grep -E "weights|test"
    for j in jobs:
        j.wait()
    got = {}
    for i in range(len(halves)):
        for l in open(f'{WORK}/bridge_D_{i}.txt'):
            m = re.match(r'test +(\d+): (\d+)/(\d+)', l)
            if m:
                got[int(m.group(1))] = 100 * int(m.group(2)) / int(m.group(3))
    print(f'ученик: {time.time() - t0:.0f} с')
    print('ученик, 20 рассказов и до 20 запрошенных, по задачам:', ' '.join(f'{k}:{v:.1f}' for k, v in sorted(got.items())))
    print(f'  среднее {mean(list(got.values())):.1f} %, задач до 95 % и выше: {sum(v >= 95 for v in got.values())} из {len(got)}')
    for e in json.load(open(f'{WORK}/transformer_D.json')):
        v = [100 * x[1] / x[2] for x in e['test']]
        print(f"трансформер, {e['size']} рассказов: среднее {mean(v):.1f} %, задач до 95 % и выше: {sum(x >= 95 for x in v)} из {len(v)}")
else:
    print('пропущено: чтобы посчитать язык, выполните %env MX_LANG=1 до первой ячейки')
